# Design — W1987881672, attempt 02

## Source definition and sample

The paper's Table 1 uses quarterly I/B/E/S earnings events from fiscal years 1984–1999 and forms annual distributions (paper physical pp. 5–6, printed pp. 225–226; Table 1 and note, physical p. 10 / printed p. 230). It requires a quarterly earnings report date and at least one analyst forecast before that date. Surprise is reported actual EPS less the single most timely individual forecast, not a consensus and not a forecast from a specified analyst. The paper excludes forecasts dated after the earnings number is known (footnote 7, physical p. 6 / printed p. 226).

The source is the catalog-listed `IBES.DET_EPSUS` (I/B/E/S U.S. Detail History EPS file with actuals). `TICKER` is the provider's persistent I/B/E/S firm key; `FPEDATS` is the fiscal-period end date; `USFIRM=1` selects the U.S. file; `MEASURE='EPS'` selects earnings per share; and `FPI` identifies a quarterly horizon. The I/B/E/S Detail History User Guide (December 2009, appendix pp. 40–41) defines `6`–`9` as quarters 1–4, `N`–`Q` as quarters 5–8, and `Y` as a quarter more than eight quarters in the future. All are retained because they denote quarter periods; no additional FPI screen is imposed. The source table has no separate historical fiscal-year label, so `year(FPEDATS)` is the operational year assigned to each quarter. This matches the paper's fiscal-year grouping as closely as the available descriptor permits; it is not proof of the original I/B/E/S fiscal-year assignment.

The screen order is:

1. Keep U.S. Detail EPS quarterly-period rows with `FPEDATS` from 1 January 1984 through 31 December 1999.
2. Require nonmissing `TICKER`, `FPEDATS`, `ACTUAL`, `ANNDATS_ACT`, `VALUE`, and `ANNDATS`.
3. Require `ANNDATS < ANNDATS_ACT`, a strictly earlier calendar date. Times do not make same-day forecasts eligible; `ANNTIMS` only ranks forecasts among eligible earlier dates.
4. For each `TICKER × FPEDATS × ANNDATS_ACT`, select the row with the latest `ANNDATS`, then latest `ANNTIMS`. At a latest-date/time tie, order by ascending analyst code, estimator code, and forecast value. The paper requires one most timely individual forecast but gives no tie rule, so this deterministic rule is a best-effort choice. No forecast-age cap is applied.
5. Retain events where `report_curr='USD'` and `CURR_ACT='USD'`. The selected-row audit found 47 records with USD report currency but CAD actual-file currency; the actual and forecast units would not be demonstrably comparable, and no approved currency-conversion input is assigned. These records are excluded after forecast selection so an earlier estimate cannot replace the most timely one. The 571 rows with `CURRFL='C'` and USD report currency remain: the guide defines `CURRFL` as the currency received by the analyst, while Detail estimates received in a different currency are converted to the company default currency. `CURR_ACT` is the currency from the Detail Actuals file and is checked against the report currency. The common-USD filter is a unit-compatibility choice, not a target-count rule.

## Variable construction and aggregation

The selected row's `ACTUAL` is reported quarterly EPS and `VALUE` is the latest individual EPS forecast. `ANNDATS_ACT` is the explicit actual announce-date field; `ACTDATS_ACT` is retained for comparison. Forecast announcement date/time are `ANNDATS`/`ANNTIMS`. `REVDATS`/`REVTIMS` are the most recent date/time an estimate was confirmed accurate (guide glossary p. 52); they are not used to decide which forecast was announced latest. The guide says Detail estimates are completely adjusted to the company-level currency and primary/diluted basis; estimate-level `CURRFL` and `PDF` describe the received basis (pp. 18, 52–53). Thus neither flag is treated as the stored forecast denomination or an eligibility screen. `PDF` and `REVDATS` are included in descriptive diagnostics only.

Surprise is `(ACTUAL - VALUE) × 100`, in cents per share. The Detail History guide (pp. 10–14 and glossary p. 52) describes the adjusted per-share product and historical split adjustments; I preserve the adjusted source values as delivered and do not round them to whole cents. Because both actual and forecast are retained only in USD, the difference has a common dollar-per-share unit. The threshold tolerance of `1e-10` cents addresses binary floating representation only; exact zero remains exact equality, with no economic rounding or near-zero smoothing. A five-case synthetic arithmetic check covers exact zero, the +3-cent cutoff, values just outside the cutoff, and the −25-cent cutoff.

Within each `year(FPEDATS)`:

- The all-earnings median uses every retained actual, including `ACTUAL=0`.
- Profit and loss medians use `ACTUAL>0` and `ACTUAL<0`; zero actuals are omitted from both.
- Zero-surprise and positive-surprise rates use all observations in the corresponding reported-profit or reported-loss group as denominator; surprises are exact zero or strictly positive.
- Small positive profit surprise is `(0, 3]` cents, conditional on a positive surprise among positive-actual observations.
- Extreme negative loss surprise is `≤−25` cents, conditional on a negative surprise among negative-actual observations.

The annual file is explicitly gridded to 1984–1999, with one record per year. It includes the quarter count, counts by actual sign, all nine annual outcomes, and date/currency diagnostics. These firm-quarter counts are distinct from each trend regression's 16 annual observations.

## Estimation

Each of the nine Table 1 outcomes is fit separately as `regress outcome fiscal_year` with a constant and conventional OLS standard error. The paper specifies univariate regressions and reports trend, slope t-value, and adjusted R-square, but it does not name the estimator or covariance convention (Table 1 note, physical p. 10 / printed p. 230). OLS and conventional inference are therefore labeled best-effort assumptions. Stata reports a signed, two-sided t-test; the paper's printed t-values are positive magnitudes even for the negative extreme-loss trend. No published estimates or sample counts enter construction.

## Date, revision, and currency limitations

The 2009 Detail History guide says its Actuals-file date was the activation date before January 1999 and the announce date thereafter (p. 16), and distinguishes announcement from activation dates in its glossary (p. 52). This source provides both `ANNDATS_ACT` and `ACTDATS_ACT`; I use the explicit announce-date variable to implement Brown's report-date condition and inspect its differences from activation date and `FPEDATS`. The older-date convention means true announcement-date semantics are not established for every historical row in this current vintage. The source has no separate version timestamp or historical release snapshot, so this review cannot recover Brown's contemporaneous vendor vintage or prove that every currently stored historical value is the original unrevised value. Restated actuals are a separate provider product; no restated series is added.

The guide documents estimate currency conversion to company default currency and notes that actuals around company-level currency changes can remain on different historical bases (pp. 11–12, 17–18, 36–39). Therefore `CURRFL` is not used as the stored estimate denomination, and a mismatching USD-report/CAD-actual event is excluded rather than silently converted. The paper does not separately document a USD-only screen, so the common-USD population is an explicit best-effort choice needed to interpret the stated cents-per-share thresholds consistently. The remaining sample is compared with the published 176,519 quarterly observations; published counts are outcomes, not targets.


In [ ]:
options errorabend mprint notes;
libname ibes "/storage/sas/base/ibes" access=readonly;
libname out "data/ibes_surprises";

/* Restrict by catalog-listed measure, region, quarter indicators, and period. */
data work.cand;
    set ibes.det_epsus(keep=TICKER CUSIP FPEDATS USFIRM MEASURE FPI PDF VALUE ACTUAL CURR CURR_ACT CURRFL report_curr ANNDATS ANNTIMS REVDATS REVTIMS ACTDATS ACTDATS_ACT ANNDATS_ACT ANNTIMS_ACT ESTIMATOR ANALYS);
    if USFIRM=1 and MEASURE='EPS' and FPI in ('6','7','8','9','N','O','P','Q','Y')
       and FPEDATS >= '01JAN1984'd and FPEDATS < '01JAN2000'd;
    fiscal_year=year(FPEDATS);
run;

/* Field availability, event-date bounds, same-day and post-report forecasts. */
proc sql;
    title 'Candidate-detail coverage and date diagnostics';
    select count(*) as candidate_rows,
           sum(not missing(TICKER)) as rows_with_ticker,
           sum(not missing(FPEDATS)) as rows_with_fpedats,
           sum(not missing(ACTUAL)) as rows_with_actual,
           sum(not missing(ANNDATS_ACT)) as rows_with_report_date,
           sum(not missing(VALUE)) as rows_with_forecast_value,
           sum(not missing(ANNDATS)) as rows_with_forecast_date,
           sum(ANNDATS < ANNDATS_ACT) as strict_pre_report_rows,
           sum(ANNDATS = ANNDATS_ACT) as same_date_rows,
           sum(ANNDATS > ANNDATS_ACT) as after_report_rows,
           min(FPEDATS) as min_fpedats format=date9.,
           max(FPEDATS) as max_fpedats format=date9.,
           min(ANNDATS_ACT) as min_report_date format=date9.,
           max(ANNDATS_ACT) as max_report_date format=date9.
      from work.cand;
quit;

proc freq data=work.cand;
    tables FPI CURRFL CURR CURR_ACT report_curr / missing;
    title 'Quarter indicators and currency fields in candidate rows';
run;

/* Keep rows meeting Brown's stated date and availability requirements. */
data work.eligible;
    set work.cand;
    if not missing(TICKER) and not missing(FPEDATS) and not missing(ACTUAL)
       and not missing(ANNDATS_ACT) and not missing(VALUE) and not missing(ANNDATS)
       and ANNDATS < ANNDATS_ACT;
    if missing(ANNTIMS) then forecast_time=0;
    else forecast_time=ANNTIMS;
    format forecast_time time8.;
    if not missing(REVDATS) then review_after_announce_days=REVDATS-ANNDATS;
    if not missing(REVDATS) then review_to_report_days=REVDATS-ANNDATS_ACT;
    surprise_cents=(ACTUAL-VALUE)*100;
run;

/* Verify event key stability before selecting one estimate per firm-quarter. */
proc sql;
    create table work.keyaudit as
    select TICKER, FPEDATS,
           count(distinct ANNDATS_ACT) as report_dates,
           count(distinct ACTUAL) as actual_values,
           count(distinct report_curr) as report_currencies,
           count(*) as eligible_rows
      from work.eligible
     group by TICKER, FPEDATS;
    title 'Eligible observation key diagnostics';
    select count(*) as eligible_firm_quarters,
           sum(report_dates>1) as multi_report_quarters,
           sum(actual_values>1) as multi_actual_quarters,
           sum(report_currencies>1) as multi_currency_quarters,
           sum(eligible_rows>1) as quarters_with_multiple_forecasts
      from work.keyaudit;
quit;

/* Count ties at the latest forecast announce date/time before deterministic tie-breaks. */
proc sql;
    create table work.latestdt as
    select TICKER, FPEDATS, ANNDATS_ACT,
           max(ANNDATS) as latest_date
      from work.eligible
     group by TICKER, FPEDATS, ANNDATS_ACT;
    create table work.latesttm as
    select l.TICKER, l.FPEDATS, l.ANNDATS_ACT, l.latest_date,
           max(e.forecast_time) as latest_time
      from work.latestdt as l inner join work.eligible as e
        on l.TICKER=e.TICKER and l.FPEDATS=e.FPEDATS
       and l.ANNDATS_ACT=e.ANNDATS_ACT and l.latest_date=e.ANNDATS
     group by l.TICKER, l.FPEDATS, l.ANNDATS_ACT, l.latest_date;
    create table work.tiesaudit as
    select e.TICKER, e.FPEDATS, e.ANNDATS_ACT, count(*) as tied_latest_rows
      from work.latesttm as l inner join work.eligible as e
        on l.TICKER=e.TICKER and l.FPEDATS=e.FPEDATS
       and l.ANNDATS_ACT=e.ANNDATS_ACT and l.latest_date=e.ANNDATS
       and l.latest_time=e.forecast_time
     group by e.TICKER, e.FPEDATS, e.ANNDATS_ACT;
    title 'Latest forecast tie diagnostics';
    select count(*) as report_events,
           sum(tied_latest_rows>1) as events_tied_at_latest_datetime,
           max(tied_latest_rows) as largest_latest_tie
      from work.tiesaudit;
quit;

/* One individual forecast per actual event; ties use ascending analyst, estimator, value. */
proc sort data=work.eligible out=work.sorted;
    by TICKER FPEDATS ANNDATS_ACT descending ANNDATS descending forecast_time ANALYS ESTIMATOR VALUE;
run;
data work.selected;
    set work.sorted;
    by TICKER FPEDATS ANNDATS_ACT;
    if first.ANNDATS_ACT;
    fiscal_year=year(FPEDATS);
    label surprise_cents='Earnings surprise (cents/share)'
          fiscal_year='Fiscal year from FPEDATS';
run;

/* Check the actuals-file denomination against the report-currency denomination. */
proc sql;
    title 'Selected-row currency and unit compatibility';
    select count(*) as selected_events,
           sum(upcase(strip(report_curr))='USD') as stored_usd_events,
           sum(upcase(strip(report_curr))='CAD') as stored_cad_events,
           sum(missing(report_curr)) as missing_stored_currency,
           sum(upcase(strip(report_curr))='USD' and upcase(strip(CURR_ACT))='USD') as usd_actual_usd_report,
           sum(upcase(strip(report_curr))='USD' and upcase(strip(CURR_ACT))='CAD') as usd_actual_cad_report,
           sum(not missing(report_curr) and upcase(strip(report_curr)) not in ('USD','CAD')) as other_stored_currency
      from work.selected;
quit;
data work.usd_selected;
    set work.selected;
    /* CURRFL identifies the currency received by the analyst, not stored VALUE.
       Require both stored VALUE/report currency and actual-file currency to be USD. */
    if upcase(strip(report_curr))='USD' and upcase(strip(CURR_ACT))='USD';
    forecast_age_days=ANNDATS_ACT-ANNDATS;
    report_period_gap_days=ANNDATS_ACT-FPEDATS;
    if not missing(ACTDATS_ACT) then report_activation_gap=ANNDATS_ACT-ACTDATS_ACT;
run;
proc freq data=work.selected;
    tables CURRFL*report_curr CURR_ACT*report_curr / missing;
    title 'Stored report currency versus received currency fields at selected rows';
run;
proc freq data=work.selected;
    tables PDF / missing;
    title 'Estimate-level primary/diluted flag (descriptive only)';
run;
proc freq data=work.selected;
    tables CURRFL*CURR_ACT / missing;
    title 'Received estimate flag versus actual-file currency';
run;
proc freq data=work.usd_selected;
    tables FPI CURRFL CURR_ACT report_curr / missing;
    title 'Currency and quarterly-period fields in common-USD sample';
run;
proc means data=work.usd_selected n nmiss min p1 p25 median p75 p99 max maxdec=5;
    var ACTUAL VALUE surprise_cents forecast_age_days report_period_gap_days report_activation_gap
        review_after_announce_days review_to_report_days;
    title 'Common-USD units, report/period alignment, forecast age, and review-date diagnostics';
run;
proc means data=work.usd_selected n min p25 median p75 p99 max maxdec=2;
    class FPI;
    var forecast_age_days report_period_gap_days;
    title 'Forecast age and report-to-period alignment by quarterly FPI';
run;
proc sql;
    title 'Date-alignment counts (diagnostic only; no date cutoff applied)';
    select count(*) as common_usd_events,
           sum(report_period_gap_days<0) as report_before_period_end,
           sum(report_period_gap_days>365) as report_over_1y_after_period_end,
           sum(report_period_gap_days>1825) as report_over_5y_after_period_end,
           sum(forecast_age_days>365) as forecast_over_1y_old,
           sum(forecast_age_days>1825) as forecast_over_5y_old,
           sum(missing(ACTDATS_ACT)) as missing_activation_date,
           sum(ANNDATS_ACT<ACTDATS_ACT) as announce_before_activation,
           sum(ANNDATS_ACT=ACTDATS_ACT) as announce_equals_activation,
           sum(missing(REVDATS)) as missing_review_date,
           sum(REVDATS>ANNDATS) as review_after_announce,
           sum(REVDATS>ANNDATS_ACT) as review_date_after_actual_report
      from work.usd_selected;
    select fiscal_year,
           count(*) as n_events,
           sum(report_period_gap_days<0) as report_before_period_end,
           sum(report_period_gap_days>365) as report_over_1y_after_period_end,
           sum(forecast_age_days>365) as forecast_over_1y_old,
           max(report_period_gap_days) as max_report_period_gap,
           max(forecast_age_days) as max_forecast_age
      from work.usd_selected
     group by fiscal_year
     order by fiscal_year;
quit;
proc sql;
    title 'Common-USD selected observations by FPEDATS calendar year';
    select fiscal_year, count(*) as n_events,
           sum(ACTUAL>0) as n_profit,
           sum(ACTUAL<0) as n_loss,
           sum(ACTUAL=0) as n_zero_actual,
           min(report_period_gap_days) as min_report_period_gap,
           max(report_period_gap_days) as max_report_period_gap,
           max(forecast_age_days) as max_forecast_age
      from work.usd_selected
     group by fiscal_year
     order by fiscal_year;
quit;

/* Boundary audit: tolerance handles only double representation, not EPS rounding. */
data work.boundary;
    input actual value exp_small exp_extreme exp_zero;
    surprise_cents=(actual-value)*100;
    small_boundary=(surprise_cents>0 and surprise_cents<=3+1e-10);
    extreme_boundary=(surprise_cents<0 and surprise_cents<=-25+1e-10);
    exact_zero=(surprise_cents=0);
    if small_boundary ne exp_small or extreme_boundary ne exp_extreme or exact_zero ne exp_zero then boundary_fail=1;
    else boundary_fail=0;
    datalines;
    0.13 0.10 1 0 0
   -0.10 0.15 0 1 0
    0.1301 0.10 0 0 0
   -0.10 0.1499 0 0 0
    0.15 0.15 0 0 1
    ;
run;
proc sql;
    title 'Exact-boundary arithmetic check';
    select count(*) as boundary_cases,
           sum(boundary_fail) as failed_boundary_cases
      from work.boundary;
quit;

/* Three annual medians, formed on each outcome's source-defined population. */
proc means data=work.usd_selected nway median noprint;
    class fiscal_year;
    var surprise_cents;
    output out=work.med_all(drop=_TYPE_ _FREQ_) median=med_all;
run;
proc means data=work.usd_selected(where=(ACTUAL>0)) nway median noprint;
    class fiscal_year;
    var surprise_cents;
    output out=work.med_profit(drop=_TYPE_ _FREQ_) median=med_profit;
run;
proc means data=work.usd_selected(where=(ACTUAL<0)) nway median noprint;
    class fiscal_year;
    var surprise_cents;
    output out=work.med_loss(drop=_TYPE_ _FREQ_) median=med_loss;
run;

/* Frequencies use the source-defined sign populations and conditional denominators. */
proc sql;
    create table work.annual_freq as
    select fiscal_year,
           count(*) as n_all,
           sum(ACTUAL>0) as n_profit,
           sum(ACTUAL<0) as n_loss,
           sum(ACTUAL=0) as n_zero_actual,
           case when sum(ACTUAL>0)>0 then 100*sum(ACTUAL>0 and surprise_cents=0)/sum(ACTUAL>0) else . end as pct_zero_profit,
           case when sum(ACTUAL<0)>0 then 100*sum(ACTUAL<0 and surprise_cents=0)/sum(ACTUAL<0) else . end as pct_zero_loss,
           case when sum(ACTUAL>0)>0 then 100*sum(ACTUAL>0 and surprise_cents>0)/sum(ACTUAL>0) else . end as pct_positive_profit,
           case when sum(ACTUAL<0)>0 then 100*sum(ACTUAL<0 and surprise_cents>0)/sum(ACTUAL<0) else . end as pct_positive_loss,
           case when sum(ACTUAL>0 and surprise_cents>0)>0 then 100*sum(ACTUAL>0 and surprise_cents>0 and surprise_cents<=3+1e-10)/sum(ACTUAL>0 and surprise_cents>0) else . end as pct_small_positive_profit,
           case when sum(ACTUAL<0 and surprise_cents<0)>0 then 100*sum(ACTUAL<0 and surprise_cents<0 and surprise_cents<=-25+1e-10)/sum(ACTUAL<0 and surprise_cents<0) else . end as pct_extreme_negative_loss
      from work.usd_selected
     group by fiscal_year;
quit;

data work.yeargrid;
    do fiscal_year=1984 to 1999;
        output;
    end;
run;
proc sort data=work.yeargrid; by fiscal_year; run;
proc sort data=work.med_all; by fiscal_year; run;
proc sort data=work.med_profit; by fiscal_year; run;
proc sort data=work.med_loss; by fiscal_year; run;
proc sort data=work.annual_freq; by fiscal_year; run;
data out.ibes_surprises;
    merge work.yeargrid(in=yearrow) work.annual_freq work.med_all work.med_profit work.med_loss;
    by fiscal_year;
    if yearrow;
    label fiscal_year='Fiscal year'
          n_all='Quarterly firm observations'
          med_all='Median earnings surprise (cents/share)'
          med_profit='Median profit surprise (cents/share)'
          med_loss='Median loss surprise (cents/share)'
          pct_zero_profit='Zero surprise among profits (percent)'
          pct_zero_loss='Zero surprise among losses (percent)'
          pct_positive_profit='Positive surprise among profits (percent)'
          pct_positive_loss='Positive surprise among losses (percent)'
          pct_small_positive_profit='Small positive profit surprise | positive surprise (percent)'
          pct_extreme_negative_loss='Extreme negative loss surprise | negative surprise (percent)';
run;

proc means data=out.ibes_surprises n nmiss min median max maxdec=4;
    var n_all med_all med_profit med_loss pct_zero_profit pct_zero_loss
        pct_positive_profit pct_positive_loss pct_small_positive_profit pct_extreme_negative_loss;
    title 'Final 16-year annual handoff checks';
run;
proc print data=out.ibes_surprises noobs;
    var fiscal_year n_all n_profit n_loss n_zero_actual med_all med_profit med_loss
        pct_zero_profit pct_zero_loss pct_positive_profit pct_positive_loss
        pct_small_positive_profit pct_extreme_negative_loss;
    title 'Annual analysis sample summary';
run;
